In [10]:
import os
import sys
import pandas as pd

# 경로 설정
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_DIR = os.path.join(BASE_DIR, "data")
sys.path.append(BASE_DIR)

# 파일 로드
gh_path = os.path.join(DATA_DIR, "code_review_gh", "data", "code_review_gh_2023.parquet")
context_path = os.path.join(DATA_DIR, "contextual_code_review", "cleaned_data.json")
reviewer_path = os.path.join(DATA_DIR, "codereviewer", "generation", "gen-train.jsonl")

print("⏳ 원본 데이터 로딩 중...")
df_gh = pd.read_parquet(gh_path)
df_context = pd.read_json(context_path)
df_reviewer = pd.read_json(reviewer_path, lines=True)
print("✅ 메모리 로드 완료!")

⏳ 원본 데이터 로딩 중...
✅ 메모리 로드 완료!


In [11]:
import importlib
import utils.adapter

# adapter.py 변경사항 강제 새로고침
importlib.reload(utils.adapter)
from utils.adapter import adapt_code_review_gh, adapt_contextual_code_review, adapt_codereviewer

print("⏳ 정제 규칙 적용 및 변환 처리 시작...\n")

# 1. code_review_gh
df_gh_adapted = adapt_code_review_gh(df_gh)
df_gh_adapted['dataset_source'] = 'code_review_gh'
print(f"1️⃣ code_review_gh: {len(df_gh):,}개 ➡️ {len(df_gh_adapted):,}개 (약 {len(df_gh) - len(df_gh_adapted):,}개 필터링됨)")

# 2. contextual_code_review
df_context_adapted = adapt_contextual_code_review(df_context)
df_context_adapted['dataset_source'] = 'contextual_code_review'
print(f"2️⃣ contextual_code_review: {len(df_context):,}개 ➡️ {len(df_context_adapted):,}개 (약 {len(df_context) - len(df_context_adapted):,}개 필터링됨)")

# 3. codereviewer
df_reviewer_adapted = adapt_codereviewer(df_reviewer)
df_reviewer_adapted['dataset_source'] = 'codereviewer'
print(f"3️⃣ codereviewer: {len(df_reviewer):,}개 ➡️ {len(df_reviewer_adapted):,}개 (약 {len(df_reviewer) - len(df_reviewer_adapted):,}개 필터링됨)")

# Unified Dataset 병합
unified_df = pd.concat([df_gh_adapted, df_context_adapted, df_reviewer_adapted], ignore_index=True)

print("\n" + "=" * 50)
print(f"🎉 최종 정제 완료된 통합 데이터셋: 총 {len(unified_df):,}개")
print("=" * 50)

⏳ 정제 규칙 적용 및 변환 처리 시작...

1️⃣ code_review_gh: 747,555개 ➡️ 630,637개 (약 116,918개 필터링됨)
2️⃣ contextual_code_review: 61,935개 ➡️ 55,276개 (약 6,659개 필터링됨)
3️⃣ codereviewer: 117,739개 ➡️ 117,739개 (약 0개 필터링됨)

🎉 최종 정제 완료된 통합 데이터셋: 총 803,652개


In [12]:
print("=" * 50)
print(f"🎉 통합 데이터셋 총 레코드 수: {len(unified_df):,}개")
print("=" * 50)
print(unified_df.groupby('dataset_source').size())
print("\n--- 통합 데이터 미리보기 ---")
unified_df[['dataset_source', 'has_issue', 'review_comment']].head(5)

🎉 통합 데이터셋 총 레코드 수: 803,652개
dataset_source
code_review_gh            630637
codereviewer              117739
contextual_code_review     55276
dtype: int64

--- 통합 데이터 미리보기 ---


,dataset_source,has_issue,review_comment
0,code_review_gh,True,Can we add gpgme as an optional dependencies. ...
1,code_review_gh,True,Can we specify lua version to luajit because t...
2,code_review_gh,True,"can we use ubuntu-22.04 here, a LTS is a bette..."
3,code_review_gh,True,Can we add GPGme as optional dependencies?
4,code_review_gh,True,"Just let it be like this, I have plan to repla..."


In [13]:
import numpy as np
import pandas as pd

# 1. 길이 계산 (Character Count)
unified_df['code_char_len'] = unified_df['source_code'].astype(str).str.len()
unified_df['diff_char_len'] = unified_df['pr_diff'].astype(str).str.len()
unified_df['comment_char_len'] = unified_df['review_comment'].astype(str).str.len()

print("==================================================")
print("📏 통합 데이터셋 길이 분포 통계 (Character Count)")
print("==================================================")

stats_df = pd.DataFrame({
    'Source Code 길이': unified_df['code_char_len'].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]),
    'PR Diff 길이': unified_df['diff_char_len'].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]),
    'Review Comment 길이': unified_df['comment_char_len'].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])
})

print(stats_df.round(1))

# 2. 대략적인 Token 수 추정 (코드/영문 기준 1 토큰 ≒ 약 3.5~4자)
print("\n" + "=" * 50)
print("💡 90% ~ 95% 커버리지를 위한 추정 토큰 수")
print("=" * 50)
p90_code = np.percentile(unified_df['code_char_len'], 90)
p95_code = np.percentile(unified_df['code_char_len'], 95)

print(f"• Source Code 90% 퍼센타일: {p90_code:,.0f} 자 (약 {p90_code/3.5:,.0f} 토큰)")
print(f"• Source Code 95% 퍼센타일: {p95_code:,.0f} 자 (약 {p95_code/3.5:,.0f} 토큰)")

📏 통합 데이터셋 길이 분포 통계 (Character Count)
       Source Code 길이  PR Diff 길이  Review Comment 길이
count        803652.0    803652.0           803652.0
mean           5157.2       846.3              191.2
std           33156.7      1092.0              353.6
min               0.0         0.0                6.0
50%             732.0       523.0               99.0
75%            1963.0      1057.0              188.0
90%            6746.0      2081.0              374.0
95%           21566.4      2803.0              648.0
99%           88399.4      3727.0             1729.0
max        14065259.0    214629.0            53234.0

💡 90% ~ 95% 커버리지를 위한 추정 토큰 수
• Source Code 90% 퍼센타일: 6,746 자 (약 1,927 토큰)
• Source Code 95% 퍼센타일: 21,566 자 (약 6,162 토큰)


In [14]:
import os

# 프로젝트 루트 기준 data 디렉토리 경로 지정
DATA_DIR = os.path.join("..", "data")
os.makedirs(DATA_DIR, exist_ok=True)  # 폴더 없으면 생성

unified_path = os.path.join(DATA_DIR, "unified_data_cleaned.parquet")

# unified_df를 파일로 저장
unified_df.to_parquet(unified_path, index=False)

print(f"💾 고품질 통합 데이터셋 저장 완료!")
print(f"• 저장 경로: {unified_path}")
print(f"• 데이터 건수: {len(unified_df):,} 개")

💾 고품질 통합 데이터셋 저장 완료!
• 저장 경로: ..\data\unified_data_cleaned.parquet
• 데이터 건수: 803,652 개


In [18]:
import importlib
import os
import sys
import pandas as pd
from dotenv import load_dotenv

# 1. 프로젝트 Root 디렉토리를 파이썬 모듈 검색 경로에 추가
sys.path.append(os.path.abspath(".."))

# 2. 프로젝트 루트의 .env 파일 로드
load_dotenv(os.path.join("..", ".env"))

# 3. scripts 및 core 모듈 불러오기
import scripts.index_to_pg as indexer

importlib.reload(indexer)

from core.db import engine  # core/db.py

# 4. 정제된 파켓 데이터 로드
DATA_DIR = os.path.join("..", "data")
unified_path = os.path.join(DATA_DIR, "unified_data_cleaned.parquet")

print("⏳ 정제된 파켓 데이터 로딩 중...")
df_clean = pd.read_parquet(unified_path)

total_count = len(df_clean)  # 대략 80만개
print(f"🎯 전체 데이터 총 {total_count:,}건 로드 완료!")

# 5. [안전 체크포인트 인덱싱 실행] 이미 들어간 건은 건너뛰고 남은 데이터만 인덱싱
print(
    f"🚀 [{total_count:,}건] 안전 체크포인트(orig_idx) 기반 OpenAI 임베딩 및 pgvector 인덱싱을 시작합니다!"
)

# 💡 embed_and_insert -> embed_and_insert_safe 로 변경되었습니다.
indexer.embed_and_insert_safe(df_clean, engine, batch_size=10, delay_seconds=2.5)

⏳ 정제된 파켓 데이터 로딩 중...
🎯 전체 데이터 총 803,652건 로드 완료!
🚀 [803,652건] 안전 체크포인트(orig_idx) 기반 OpenAI 임베딩 및 pgvector 인덱싱을 시작합니다!


AttributeError: module 'scripts.index_to_pg' has no attribute 'embed_and_insert_safe'

In [8]:
import sys
import os
from sqlalchemy import text
from openai import OpenAI
from dotenv import load_dotenv

# 1. 환경 변수 및 DB 엔진 로드
sys.path.append(os.path.abspath(".."))
load_dotenv(os.path.join("..", ".env"))

from core.db import engine

client = OpenAI()

# 2. 유사도 검색 함수
def search_similar_code_reviews(query_text: str, top_k: int = 3):
    # 입력 질의문 임베딩
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=[query_text]
    )
    query_vector = response.data[0].embedding

    # ::vector 캐스팅 대신 CAST(... AS vector) 문법으로 구문 오류 수정
    search_sql = text("""
        SELECT 
            dataset_source,
            source_code,
            pr_diff,
            review_comment,
            has_issue,
            1 - (embedding <=> CAST(:query_vector AS vector)) AS similarity
        FROM code_reviews
        ORDER BY embedding <=> CAST(:query_vector AS vector) ASC
        LIMIT :top_k;
    """)

    with engine.connect() as conn:
        results = conn.execute(
            search_sql, 
            {"query_vector": str(query_vector), "top_k": top_k}
        ).fetchall()

    print("=" * 65)
    print(f"🔍 입력 질의문(Query): '{query_text}'")
    print("=" * 65)

    for idx, row in enumerate(results, 1):
        print(f"\n🏆 [TOP {idx}] 유사도 점수(Similarity): {row.similarity:.4f}")
        print(f"📌 출처: {row.dataset_source} | 이슈 여부: {row.has_issue}")
        print(f"💬 리뷰 코멘트:\n   └─ {row.review_comment}")
        print(f"📝 관련 PR Diff (미리보기):")
        diff_preview = str(row.pr_diff)[:200].replace('\n', '\n   ')
        print(f"   {diff_preview}...")
        print("-" * 65)

# 3. 테스트 실행!
search_similar_code_reviews("Check if list or array is null or empty before iterating", top_k=3)

🔍 입력 질의문(Query): 'Check if list or array is null or empty before iterating'

🏆 [TOP 1] 유사도 점수(Similarity): 0.3874
📌 출처: contextual_code_review | 이슈 여부: True
💬 리뷰 코멘트:
   └─ `.isBlank()` will capture cases such as "      "(empty spaces) as well. Do we really need this or just would `isEmpty()` is suffcient?
📝 관련 PR Diff (미리보기):
   ...
-----------------------------------------------------------------

🏆 [TOP 2] 유사도 점수(Similarity): 0.3514
📌 출처: contextual_code_review | 이슈 여부: True
💬 리뷰 코멘트:
   └─ If there are no checkpoints to list, should we be returning an empty array rather than an exception?
📝 관련 PR Diff (미리보기):
   --- before
   +++ after
   @@ -1,25 +1,27 @@
    public Flux<Checkpoint> listCheckpoints(String fullyQualifiedNamespace, String eventHubName, String consumerGroup) {
    String prefix = prefixBuilder(fullyQualifi...
-----------------------------------------------------------------

🏆 [TOP 3] 유사도 점수(Similarity): 0.3432
📌 출처: contextual_code_review | 이슈 여부: True
💬 리뷰 코멘트:
   

In [15]:
from openai import OpenAI

client = OpenAI()

def generate_rag_code_review(user_diff_or_query: str, top_k: int = 3):
    # 1단계: Vector DB에서 관련 과거 리뷰 검색 (Retrieval)
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=[user_diff_or_query]
    )
    query_vector = response.data[0].embedding

    search_sql = text("""
        SELECT pr_diff, review_comment, 1 - (embedding <=> CAST(:query_vector AS vector)) AS similarity
        FROM code_reviews
        ORDER BY embedding <=> CAST(:query_vector AS vector) ASC
        LIMIT :top_k;
    """)

    with engine.connect() as conn:
        retrieved_docs = conn.execute(search_sql, {"query_vector": str(query_vector), "top_k": top_k}).fetchall()

    # 2단계: Retrieve된 과거 맥락(Context) 조립
    context_text = ""
    for idx, doc in enumerate(retrieved_docs, 1):
        context_text += f"[과거 유사 리뷰 예시 {idx}]\n"
        context_text += f"리뷰 코멘트: {doc.review_comment}\n"
        context_text += f"관련 Diff: {str(doc.pr_diff)[:150]}\n\n"

    # 3단계: LLM 프롬프트 구성 (System Prompt + RAG Context + User Query)
    prompt = f"""
너는 깃허브(GitHub)의 꼼꼼하고 친절한 Senior Software Engineer야.
개발자가 작성한 코드/질문에 대해 과거 코드 리뷰 선례(Context)를 참고해서 건설적이고 명확한 코드 리뷰 의견을 작성해줘.

[과거 유사 레퍼런스 데이터]
{context_text}

[검토할 질문 또는 코드]
{user_diff_or_query}

위 레퍼런스와 질문을 바탕으로, 개발자에게 전달할 **최종 Code Review Comment**를 한글과 보편적인 영문 코드로 작성해줘.
"""

    # 4단계: OpenAI LLM 호출 (Generation)
    completion = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are an expert Code Reviewer AI."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2
    )

    print("==================================================")
    print("🤖 RAG 기반 AI Senior Reviewer의 최종 피드백")
    print("==================================================")
    print(completion.choices[0].message.content)

# 실행 테스트!
generate_rag_code_review("Check if list or array is null or empty before iterating")

🤖 RAG 기반 AI Senior Reviewer의 최종 피드백
**최종 Code Review Comment:**

리뷰 코멘트: 리스트나 배열이 null이거나 비어있는 경우를 확인한 후 반복(iterate)하는 것이 좋습니다. 이를 통해 NullPointerException과 같은 예외를 방지할 수 있습니다. 예를 들어, 다음과 같이 조건문을 추가하여 안전성을 높일 수 있습니다:

```java
if (list != null && !list.isEmpty()) {
    for (Item item : list) {
        // 반복 처리
    }
}
```

이렇게 하면 코드의 안정성을 높이고, 예외 발생 가능성을 줄일 수 있습니다. 

---

Review Comment: It is advisable to check if the list or array is null or empty before iterating. This helps prevent exceptions like NullPointerException. For instance, you can enhance safety by adding a condition like this:

```java
if (list != null && !list.isEmpty()) {
    for (Item item : list) {
        // Processing logic
    }
}
```

This will improve the robustness of the code and reduce the likelihood of exceptions.


In [14]:
import importlib
from core.db import engine
import scripts.index_to_pg as indexer

# 1. 임의의 데이터 10건 추출 (Random Sampling)
test_sample_df = df_clean.sample(n=10, random_state=42).reset_index(drop=True)

print("🧪 [Smoke Test] 임의의 10건 데이터 파이프라인 검증 시작...")

try:
    # 2. 파이프라인(토큰 자르기 -> 임베딩 API -> pgvector Bulk Insert) 통과
    importlib.reload(indexer)
    indexer.embed_and_insert(
        test_sample_df, engine, batch_size=5, delay_seconds=1.0
    )
    print("\n✅ [테스트 성공] 10건 데이터가 에러 없이 성공적으로 저장되었습니다!")

except Exception as e:
    print(f"\n❌ [테스트 실패] 에러 발생: {e}")

🧪 [Smoke Test] 임의의 10건 데이터 파이프라인 검증 시작...
✅ pgvector 확장 및 DB 테이블(code_review_vectors) 준비 완료!
🚀 총 10건 데이터 임베딩 및 DB 저장 시작 (Batch Size: 5, Interval: 1.0s)...
  - [5/10] 건 저장 완료
  - [10/10] 건 저장 완료
🎉 모든 데이터 저장 완료!

✅ [테스트 성공] 10건 데이터가 에러 없이 성공적으로 저장되었습니다!


In [17]:
import pandas as pd
from sqlalchemy import text
from core.db import engine

# 1. 원본 데이터 로드
df_clean = pd.read_parquet("../data/unified_data_cleaned.parquet")

# 텍스트 공백 정제 함수
def clean_text(val):
    return str(val).strip() if val is not None else ""

# 원본 텍스트 매핑 키 생성 (strip 적용으로 미세한 공백 차이 무시)
df_clean['mapping_key'] = (
    df_clean['pr_diff'].apply(clean_text) + "|||" + df_clean['review_comment'].apply(clean_text)
)
text_to_orig_idx = {key: idx for idx, key in enumerate(df_clean['mapping_key'])}

# 2. DB에서 orig_idx가 NULL인 기존 데이터 가져오기
with engine.connect() as conn:
    rows = conn.execute(
        text("SELECT id, pr_diff, review_comment FROM code_review_vectors WHERE orig_idx IS NULL")
    ).fetchall()

print(f"🔍 매핑 대상 DB 레코드: {len(rows):,}건")

# 3. 정확한 원본 index 매핑 업데이트 생성
update_data = []
for r in rows:
    db_id = r[0]
    db_key = clean_text(r[1]) + "|||" + clean_text(r[2])
    
    if db_key in text_to_orig_idx:
        real_orig_idx = text_to_orig_idx[db_key]
        update_data.append({"p_id": db_id, "p_orig_idx": real_orig_idx})

print(f"✅ 매핑 성공: {len(update_data):,}건")

# 4. DB 일괄 반영 (빈 리스트 실행 에러 방지)
if update_data:
    with engine.begin() as conn:
        conn.execute(
            text("UPDATE code_review_vectors SET orig_idx = :p_orig_idx WHERE id = :p_id"),
            update_data
        )
    print("🎉 기존 데이터의 orig_idx 보정이 성공적으로 완료되었습니다!")
else:
    print("⚠️ 매핑된 데이터가 없습니다. 이미 모든 orig_idx가 채워져 있는지 확인해 주세요.")

🔍 매핑 대상 DB 레코드: 0건
✅ 매핑 성공: 0건
⚠️ 매핑된 데이터가 없습니다. 이미 모든 orig_idx가 채워져 있는지 확인해 주세요.
